# Notebook 06: Ray Train Distributed

## Why This Matters

Ray Train is the bridge between Ray's cluster management and PyTorch's distributed training. It handles the boilerplate that trips up most engineers: setting up process groups across nodes, managing distributed samplers, checkpointing fault-tolerant training runs, and reporting metrics back from all workers. Understanding Ray Train means you can take a single-GPU PyTorch script and scale it to hundreds of GPUs without rewriting your training logic.

In [ ]:
%pip install -q 'ray[train]' torch numpy matplotlib

In [ ]:
import ray
import ray.train as train
from ray.train import ScalingConfig, RunConfig, CheckpointConfig
from ray.train.torch import TorchTrainer, prepare_model, prepare_data_loader
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import os

if ray.is_initialized():
    ray.shutdown()
ray.init(num_cpus=4, ignore_reinit_error=True)

print(f"Ray version: {ray.__version__}")
print(f"PyTorch version: {torch.__version__}")
print("Ready.")

## 1. Ray Train Architecture

Ray Train abstracts distributed training into three components:

**Trainer**: the top-level object that configures and launches training. You pass it:
- A `train_loop_per_worker` function (your training code)
- A `ScalingConfig` (how many workers, GPUs per worker)
- A `RunConfig` (checkpointing, logging)

**Workers**: Ray actors running your `train_loop_per_worker`. Ray Train automatically:
- Sets up `torch.distributed` process groups across workers
- Wraps your model with DDP
- Patches your DataLoader to use `DistributedSampler`
- Handles NCCL backend selection

**Driver**: your script, which calls `trainer.fit()` and blocks until training completes.

The key design principle: your training loop is a regular PyTorch loop. You add one `prepare_model()` and `prepare_data_loader()` call, and Ray handles the distribution.

In [ ]:
# Ray Train: define the training loop
def train_loop_per_worker(config):
    """This function runs on EACH worker in the Ray Train job."""
    import torch
    import torch.nn as nn
    from ray.train.torch import prepare_model, prepare_data_loader
    import ray.train as train
    
    # Hyperparameters from config
    lr = config.get("lr", 0.01)
    batch_size = config.get("batch_size", 64)
    n_epochs = config.get("n_epochs", 5)
    
    # Build model (same on all workers -- DDP will sync weights)
    model = nn.Sequential(
        nn.Linear(32, 128),
        nn.ReLU(),
        nn.Linear(128, 64),
        nn.ReLU(),
        nn.Linear(64, 10)
    )
    
    # Wrap with DDP -- Ray Train handles this automatically
    model = prepare_model(model)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    # Synthetic dataset
    torch.manual_seed(42)
    dataset = torch.utils.data.TensorDataset(
        torch.randn(1000, 32),
        torch.randint(0, 10, (1000,))
    )
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    # Wrap loader -- adds DistributedSampler automatically
    loader = prepare_data_loader(loader)
    
    # Training loop
    for epoch in range(n_epochs):
        model.train()
        total_loss = 0.0
        n_batches = 0
        for X, y in loader:
            optimizer.zero_grad()
            logits = model(X)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1
        
        avg_loss = total_loss / n_batches
        
        # Report metrics back to driver (from all workers)
        train.report({"epoch": epoch, "loss": avg_loss})

# Run with 2 workers (simulating 2-GPU training on CPU)
print("Launching Ray Train job...")
trainer = TorchTrainer(
    train_loop_per_worker=train_loop_per_worker,
    train_loop_config={"lr": 0.01, "batch_size": 32, "n_epochs": 3},
    scaling_config=ScalingConfig(num_workers=2, use_gpu=False),
)

result = trainer.fit()
print(f"\nTraining complete!")
print(f"Final metrics: {result.metrics}")
print(f"Best checkpoint: {result.checkpoint}")

## 2. ScalingConfig: GPU and CPU Resource Allocation

`ScalingConfig` is how you tell Ray Train what resources to allocate per worker:

| Config | Meaning | Example |
|--------|---------|---------|
| `num_workers=N` | N training worker processes | 8 for 8-GPU job |
| `use_gpu=True` | Each worker gets 1 GPU | Standard for GPU training |
| `resources_per_worker={"GPU": 2}` | Each worker gets 2 GPUs | For tensor parallelism |
| `trainer_resources={"CPU": 1}` | Resources for the driver | Usually minimal |
| `placement_strategy="PACK"` | Pack workers on fewest nodes | Better NVLink/PCIe BW |
| `placement_strategy="SPREAD"` | Spread across nodes | Better fault tolerance |

The `PACK` strategy is almost always better for GPU training: workers on the same node communicate via NVLink (600 GB/s) instead of InfiniBand (200 GB/s). Only use `SPREAD` if you have specific fault-tolerance requirements.

In [ ]:
# ScalingConfig examples
from ray.train import ScalingConfig

configs = {
    "Single GPU": ScalingConfig(num_workers=1, use_gpu=False),
    "8x A100 (1 node)": ScalingConfig(
        num_workers=8, 
        use_gpu=True,
        placement_strategy="PACK"  # all on same node -> NVLink
    ),
    "64x A100 (8 nodes)": ScalingConfig(
        num_workers=64,
        use_gpu=True,
        placement_strategy="PACK"  # pack 8 per node
    ),
    "Tensor parallel (2 GPUs/worker)": ScalingConfig(
        num_workers=4,
        use_gpu=True,
        resources_per_worker={"GPU": 2}  # each worker has 2 GPUs
    ),
}

for name, sc in configs.items():
    print(f"{name}:")
    print(f"  num_workers: {sc.num_workers}")
    print(f"  use_gpu: {sc.use_gpu}")
    print()

print("Key rule: ScalingConfig.num_workers == world_size for DDP/FSDP")
print("Ray Train automatically sets WORLD_SIZE, RANK, LOCAL_RANK env vars")

## 3. Checkpointing and Fault Tolerance

Long training runs (weeks for LLMs) require robust checkpointing. Ray Train provides:

**Automatic checkpointing**: call `train.report(metrics, checkpoint=Checkpoint.from_dict(state))` and Ray saves the checkpoint. On failure, resume from the latest checkpoint automatically.

**CheckpointConfig**: configure how many checkpoints to keep, whether to keep the best by metric, etc.

**Fault tolerance**: `RunConfig(failure_config=FailureConfig(max_failures=3))` tells Ray to restart failed workers up to 3 times before giving up.

The checkpoint flow:
1. Worker calls `train.report(metrics, checkpoint=ckpt)`
2. Ray serializes and stores the checkpoint (local disk or cloud)
3. On worker failure, Ray restarts from the latest checkpoint
4. After training, `result.checkpoint` points to the final checkpoint

In [ ]:
import tempfile
from ray.train import Checkpoint, CheckpointConfig, RunConfig
from ray.train.torch import TorchTrainer

def train_with_checkpointing(config):
    import torch
    import torch.nn as nn
    from ray.train.torch import prepare_model
    import ray.train as train
    
    model = nn.Linear(16, 4)
    model = prepare_model(model)
    optimizer = torch.optim.SGD(model.parameters(), lr=config.get("lr", 0.01))
    
    # Load checkpoint if resuming
    checkpoint = train.get_checkpoint()
    start_epoch = 0
    if checkpoint:
        with checkpoint.as_directory() as ckpt_dir:
            state = torch.load(os.path.join(ckpt_dir, "state.pt"), weights_only=True)
            model.module.load_state_dict(state["model"])
            optimizer.load_state_dict(state["optimizer"])
            start_epoch = state["epoch"] + 1
            print(f"Resuming from epoch {start_epoch}")
    
    for epoch in range(start_epoch, config.get("n_epochs", 5)):
        # Simulate a training step
        X = torch.randn(32, 16)
        y = torch.randint(0, 4, (32,))
        loss = nn.CrossEntropyLoss()(model(X), y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
        # Save checkpoint every epoch
        with tempfile.TemporaryDirectory() as tmpdir:
            torch.save({
                "model": model.module.state_dict(),
                "optimizer": optimizer.state_dict(),
                "epoch": epoch,
            }, os.path.join(tmpdir, "state.pt"))
            checkpoint = Checkpoint.from_directory(tmpdir)
            train.report({"epoch": epoch, "loss": loss.item()}, checkpoint=checkpoint)

trainer = TorchTrainer(
    train_loop_per_worker=train_with_checkpointing,
    train_loop_config={"n_epochs": 4, "lr": 0.05},
    scaling_config=ScalingConfig(num_workers=2, use_gpu=False),
    run_config=RunConfig(
        checkpoint_config=CheckpointConfig(
            num_to_keep=2,              # keep last 2 checkpoints
            checkpoint_score_attribute="loss",
            checkpoint_score_order="min",  # keep best (lowest loss)
        )
    ),
)

result = trainer.fit()
print(f"Training done. Final metrics: {result.metrics}")
print(f"Best checkpoint path: {result.best_checkpoints[0][0] if result.best_checkpoints else 'N/A'}")

## 4. Ray Train vs Manual DDP Setup

Comparing the two approaches:

| Aspect | Manual DDP (`mp.spawn`) | Ray Train |
|--------|------------------------|-----------|
| Process group setup | Manual `init_process_group` | Automatic |
| DistributedSampler | Manual | Automatic via `prepare_data_loader` |
| Multi-node | Complex SSH setup | Ray cluster handles it |
| Fault tolerance | None (crash = restart from scratch) | Automatic checkpoint/resume |
| Metric reporting | Manual aggregation | `train.report()` from all workers |
| GPU affinity | Manual `device_ids` | Automatic |
| Experiment tracking | Manual | Integrates with MLflow, W&B, Comet |

For production training jobs: always use Ray Train (or a comparable managed service). The fault-tolerance and multi-node support alone justify the slight overhead.

In [ ]:
# Side-by-side comparison
manual_ddp_code = '''
# MANUAL DDP SETUP (error-prone, fragile)
import os, torch.distributed as dist, torch.multiprocessing as mp

def worker(rank, world_size):
    os.environ["MASTER_ADDR"] = "localhost"  # hardcoded -- breaks multi-node!
    os.environ["MASTER_PORT"] = "12355"
    dist.init_process_group("nccl", rank=rank, world_size=world_size)
    
    model = MyModel().cuda(rank)
    model = DDP(model, device_ids=[rank])
    
    # Must manually handle DistributedSampler
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank)
    loader = DataLoader(dataset, sampler=sampler)
    
    # No fault tolerance, no checkpointing coordination
    for epoch in range(n_epochs):
        for batch in loader:
            ...  # training
    
    dist.destroy_process_group()

mp.spawn(worker, args=(8,), nprocs=8)
'''

ray_train_code = '''
# RAY TRAIN (clean, production-ready)
from ray.train.torch import TorchTrainer

def train_loop(config):
    model = prepare_model(MyModel())  # DDP wrapping automatic
    loader = prepare_data_loader(DataLoader(dataset))  # sampler automatic
    
    for epoch in range(config["n_epochs"]):
        for batch in loader:
            ...  # same training code
        train.report({"loss": loss}, checkpoint=save_checkpoint())

trainer = TorchTrainer(
    train_loop,
    scaling_config=ScalingConfig(num_workers=8, use_gpu=True),
    run_config=RunConfig(failure_config=FailureConfig(max_failures=3)),
)
result = trainer.fit()  # handles multi-node, fault tolerance, metrics
'''

print("=== Manual DDP ===")
print(manual_ddp_code)
print("=== Ray Train ===")
print(ray_train_code)

## Summary

### Key Concepts

| Component | Role | Key API |
|-----------|------|---------|
| `TorchTrainer` | Launch distributed training | `trainer.fit()` |
| `train_loop_per_worker` | Your training code, runs on each worker | Regular PyTorch |
| `prepare_model` | Wrap with DDP automatically | `model = prepare_model(model)` |
| `prepare_data_loader` | Add DistributedSampler | `loader = prepare_data_loader(loader)` |
| `train.report` | Send metrics + checkpoint to driver | `train.report({"loss": l}, checkpoint=ckpt)` |
| `ScalingConfig` | Configure workers and GPUs | `ScalingConfig(num_workers=8, use_gpu=True)` |
| `CheckpointConfig` | Checkpoint retention policy | `num_to_keep=2, score_attribute="loss"` |

**The 3-line migration from manual DDP to Ray Train**:
1. Replace `mp.spawn(...)` with `TorchTrainer(...).fit()`
2. Add `model = prepare_model(model)` and `loader = prepare_data_loader(loader)`
3. Add `train.report(metrics, checkpoint=ckpt)` at end of epoch

**Next**: `07_tensor_parallelism.ipynb` -- splitting individual layers across GPUs for models too large for FSDP.